In [49]:
import pandas as pd

df = pd.read_csv('/content/beer-servings (2).csv')
display(df.head())

,Unnamed: 0,country,beer_servings,spirit_servings,wine_servings,total_litres_of_pure_alcohol,continent
0,0,Afghanistan,0.0,0.0,0.0,0.0,Asia
1,1,Albania,89.0,132.0,54.0,4.9,Europe
2,2,Algeria,25.0,0.0,14.0,0.7,Africa
3,3,Andorra,245.0,138.0,312.0,12.4,Europe
4,4,Angola,217.0,57.0,45.0,5.9,Africa


In [50]:
missing_values = df.isnull().sum()
display(missing_values)

,0
Unnamed: 0,0
country,0
beer_servings,8
spirit_servings,8
wine_servings,6
total_litres_of_pure_alcohol,1
continent,0


In [62]:
df = df.drop(columns=['Unnamed: 0'])
display(df.head())

,country,beer_servings,spirit_servings,wine_servings,total_litres_of_pure_alcohol,continent
0,Afghanistan,0.0,0.0,0.0,0.0,Asia
1,Albania,89.0,132.0,54.0,4.9,Europe
2,Algeria,25.0,0.0,14.0,0.7,Africa
3,Andorra,245.0,138.0,312.0,12.4,Europe
4,Angola,217.0,57.0,45.0,5.9,Africa


In [63]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score

In [64]:
df = df.dropna(subset=["total_litres_of_pure_alcohol"])

In [65]:
X = df.drop("total_litres_of_pure_alcohol", axis=1)
y = df["total_litres_of_pure_alcohol"]

In [70]:
categorical_features = ["country", "continent"]
numerical_features = ["beer_servings", "spirit_servings", "wine_servings"]

In [71]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [72]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [73]:
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

lr_pipeline.fit(X_train, y_train)

y_pred_lr = lr_pipeline.predict(X_test)

r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression R2 Score:", r2_lr)

Linear Regression R2 Score: 0.9439890388715062


In [74]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)

r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest R2 Score:", r2_rf)

Random Forest R2 Score: 0.8764647270778655


In [75]:
param_grid = {
    "model__n_estimators": [50,100,200],
    "model__max_depth": [None,5,10],
    "model__min_samples_split": [2,5]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring="r2"
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer())]),
                                                                         ['beer_servings',
                                                                          'spirit_servings',
                                                                          'wine_servings']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('encoder',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['country',
                                                                          'continent'])])),
                                       ('model',
                                        RandomForestRegressor(random_state=42))]),
             param_grid={'model__max_depth': [None, 5, 10],
                         'model__min_samples_split': [2, 5],
                         'model__n_estimators': [50, 100, 200]},
             scoring='r2')

In [76]:
print("Best Parameters:", grid_search.best_params_)

best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)

r2_best = r2_score(y_test, y_pred_best)

print("Tuned Random Forest R2:", r2_best)

Best Parameters: {'model__max_depth': 10, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Tuned Random Forest R2: 0.874673193559707


In [77]:
print("Linear Regression R2:", r2_lr)
print("Random Forest R2:", r2_rf)
print("Tuned Random Forest R2:", r2_best)

Linear Regression R2: 0.9439890388715062
Random Forest R2: 0.8764647270778655
Tuned Random Forest R2: 0.874673193559707


In [78]:
import pickle

pickle.dump(best_model, open("alcohol_model.pkl","wb"))

In [79]:
import joblib

joblib.dump(best_model, "alcohol_model.pkl")

['alcohol_model.pkl']